Ссылка на урок: [Спринт 2/5: 2 RAG: от сырых данных до точных ответов → Тема 3/4: Архитектура RAG → Урок 4/6: Трансформация запросов: query expansion и HyDE]( https://practicum.yandex.ru/learn/ai-engineer/courses/99ad0dee-d9e6-4da1-b921-64e7e9653a4e/sprints/1104074/topics/b36d9f5f-3f26-4fbb-bd65-71d33c4523ef/lessons/3abd647b-1411-43ab-bba0-e380c44063e9/#b4867967-36e5-48bd-9834-385950162dae )

In [1]:
from openai import OpenAI

# Клиент ходит на локальный сервер llama.cpp вместо облака
client = OpenAI(base_url="http://localhost:8080/v1", api_key="not-needed")

# Имя модели сервер игнорирует — она уже загружена в него при старте
# temperature=0 делает вывод воспроизводимым
# enable_thinking=False отключает режим рассуждения Qwen3.5
resp = client.chat.completions.create(
    model="qwen3.5",
    messages=[{"role": "user", "content": "Перефразируй: льготы для ИП"}],
    temperature=0.0,
    max_tokens=128,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}})

print(resp.choices[0].message.content)

Вот несколько вариантов перефразирования фразы «льготы для ИП», в зависимости от того, какой контекст вам нужен:

**Официально-деловой стиль:**
*   Предоставляемые государством преференции индивидуальным предпринимателям.
*   Специальные условия налогообложения и поддержки для самозанятых и ИП.
*   Система стимулов для развития малого бизнеса в лице индивидуальных предпринимателей.

**Лаконичный/Краткий стиль:**
*   Поддержка ИП.
*   Выгоды для предприн


In [3]:
import json
from pathlib import Path

import numpy as np
from sentence_transformers import SentenceTransformer

def find_corpus2() -> Path:
    cwd = Path.cwd().resolve()
    candidates = []

    for base in [cwd, *cwd.parents]:
        candidates.extend([
            base / "corpus2.json",
            base / "query-expansion-and-hy-de" / "corpus2.json",
        ])

    for candidate in candidates:
        if candidate.exists():
            return candidate

    raise FileNotFoundError(f"Не найден corpus2.json. Текущая папка: {cwd}")

corpus_path = find_corpus2()
print("Используется corpus:", corpus_path)

with corpus_path.open(encoding="utf-8") as f:
    data = json.load(f)
chunks = data["chunks"]      # list[dict]: 84 чанка, поля id, category, text
queries = data["queries"]    # list[dict]: 24 запроса, поля query и relevant_ids
texts = [c["text"] for c in chunks]

print("Чанков:", len(chunks), "| Запросов:", len(queries))
print("Пример запроса:", queries[18])     # Короткий разговорный запрос
print("Пример чанка:", chunks[54])         # Формальный документ-ответ на него

# Вывод:
# Чанков: 84 | Запросов: 24
# Пример запроса: {'query': 'как копятся бонусы', 'relevant_ids': [54]}
# Пример чанка: {'id': 54, 'category': 'лояльность', 'text': 'Баллы программы лояльности зачисляются за каждую оплаченную покупку и уменьшают сумму последующих заказов.'}

# --- Эмбеддер: тот же multilingual-e5, что и в теме (на GPU) ---
embedder = SentenceTransformer("intfloat/multilingual-e5-base", device="cuda")

# Документы кодируем заранее с префиксом passage: — предвычисление индекса
doc_vecs = embedder.encode(["passage: " + t for t in texts],
                           normalize_embeddings=True, show_progress_bar=False)


def encode_query(text):
    """Кодирует текст как запрос (префикс query:), вектор нормализован."""
    return embedder.encode("query: " + text, normalize_embeddings=True)


def search(text, n=50):
    """Поиск по тексту: кодирует его как query: и возвращает индексы
    top-n документов по убыванию косинусного сходства."""
    return list(np.argsort(encode_query(text) @ doc_vecs.T)[::-1][:n])


def search_by_vector(vec, k=5):
    """Поиск по готовому вектору (для HyDE): нормирует его и возвращает top-k."""
    vec = vec / np.linalg.norm(vec)
    return list(np.argsort(vec @ doc_vecs.T)[::-1][:k])

Используется corpus: C:\PythonProg\all-agent.rag-arch\query-expansion-and-hy-de\corpus2.json
Чанков: 84 | Запросов: 24
Пример запроса: {'query': 'как копятся бонусы', 'relevant_ids': [54]}
Пример чанка: {'id': 54, 'category': 'лояльность', 'text': 'Баллы программы лояльности зачисляются за каждую оплаченную покупку и уменьшают сумму последующих заказов.'}


In [4]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8080/v1", api_key="not-needed")
LLM = "qwen3.5"                                       # Имя сервер игнорирует
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}


def generate(messages, max_tokens=160):
    """Один вызов локальной LLM. temperature=0 — для воспроизводимости,
    режим рассуждения Qwen3.5 отключён, поэтому сразу приходит готовый текст."""
    resp = client.chat.completions.create(
        model=LLM, messages=messages, temperature=0.0,
        max_tokens=max_tokens, extra_body=NO_THINK)
    return resp.choices[0].message.content.strip()

In [5]:
from collections import defaultdict
# search(), generate(), texts, queries — из общего кода


def expand(query, n=5):
    """Query expansion: генерируем n перефразировок запроса официальными
    словами базы знаний. Возвращаем список строк-перефразировок."""
    text = generate([
        {"role": "system", "content": "Ты помощник поиска по базе знаний "
                                      "интернет-магазина."},
        {"role": "user", "content": f"Переформулируй запрос клиента {n} способами "
         f"официальными словами из базы знаний. Используй полные фразы (не одно "
         f"слово), синонимы и термины. По одной формулировке на строку, без "
         f"нумерации.\n\nЗапрос: {query}"}])
    # Пропуск 1: разбейте text на строки, уберите пустые, верните первые n
    return [line.strip() for line in text.splitlines() if line.strip()][:n]


def rrf(ranked_lists, k=60, top_k=5):
    """Reciprocal Rank Fusion: сливаем несколько выдач по рангам.
    ranked_lists — список выдач (каждая — список индексов документов)."""
    scores = defaultdict(float)
    for lst in ranked_lists:
        for rank, doc in enumerate(lst, start=1):
            # Пропуск 2: добавьте документу doc вклад 1 / (k + rank)
            scores[doc] += 1.0 / (k + rank)
    ranked = sorted(scores.items(), key=lambda x: -x[1])
    return [doc for doc, _ in ranked[:top_k]]


# --- Прогон на запросах, где обычный поиск промахивается ---
for qi in [16, 18]:
    query = queries[qi]["query"]
    rel = set(queries[qi]["relevant_ids"])
    paras = expand(query)
    # Пропуск 3: список выдач — поиск по query и по каждой перефразировке;
    #            слейте их через rrf() в топ-5
    expanded = rrf([search(query)] + [search(p) for p in paras])
    mark = lambda i: "+" if i in rel else " "
    print(f"\n[{qi}] {query}")
    for p in paras:
        print("   ·", p)
    print("  обычный поиск top5:",
          [f"{mark(i)}#{i} {texts[i][:32]}" for i in search(query)[:5]])
    print("  query expansion top5:",
          [f"{mark(i)}#{i} {texts[i][:32]}" for i in expanded[:5]])


[16] как отписаться от платежей
   · Как прекратить автоматические списания средств с моей банковской карты.
   · Как отменить подписку на автоматические платежи в вашем магазине.
   · Как остановить регулярные списания средств с моего счета.
   · Как удалить подписку на автоматические транзакции.
   · Как прекратить автоматические списания средств с моей карты.
  обычный поиск top5: [' #49 Отписаться от новостной рассылки', ' #22 Отписаться от рекламной рассылки', ' #46 Оплата подписки списывается с ка', ' #45 Рассчитаться в момент вручения к', ' #50 Платежи по рассрочке списываются']
  query expansion top5: [' #46 Оплата подписки списывается с ка', ' #49 Отписаться от новостной рассылки', '+#48 Регулярные списания прекращаются', ' #50 Платежи по рассрочке списываются', ' #22 Отписаться от рекламной рассылки']

[18] как копятся бонусы
   · Каков порядок начисления бонусных баллов?
   · Как происходит накопление баллов лояльности?
   · Какие условия действуют при формировании бонусног